## Orders Table - PySpark Cleaning

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, lit, to_date
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("OrdersCleaning").getOrCreate()

df_orders = spark.read.csv("../Coursework_dataset/orders.csv", header=True, inferSchema=True)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/14 18:20:17 WARN Utils: Your hostname, saag, resolves to a loopback address: 127.0.1.1; using 192.168.0.102 instead (on interface wlp1s0)
26/04/14 18:20:17 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/14 18:20:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/14 18:20:21 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/14 18:20:21 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/04/14 18:20:21 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/04/14 18:20:21 WARN Utils: Servic

### Preview the orders table

In [2]:
df_orders.show(5)

+----------+-----------+----------+-------+------------+----------------+-------------+----------------+------------+
|  order_id|customer_id|order_date|bs_year|order_status|  payment_method|shipping_city|shipping_country|discount_pct|
+----------+-----------+----------+-------+------------+----------------+-------------+----------------+------------+
|ord_000001|cust_049923|2023-11-18|   2080|   Cancelled|     Connect IPS|      Hetauda|           Nepal|        0.02|
|ord_000002|cust_008684|2023-05-17|   2080|     Shipped|     Connect IPS|      Kolkata|           India|        0.22|
|ord_000003|cust_067594|2022-07-10|   2079|    Returned|Cash on Delivery|      Pokhara|           Nepal|        0.21|
|ord_000004|cust_055649|2023-07-25|   2080|   Delivered|Cash on Delivery|       Butwal|           Nepal|        0.05|
|ord_000005|cust_038657|2020-09-13|   2077|   Delivered|          Khalti|      Birgunj|           Nepal|        0.02|
+----------+-----------+----------+-------+------------+

### Check how many values each column has (non-null count)

In [3]:
non_null_counts = df_orders.select([
    count(when(col(c).isNotNull(), c)).alias(c) for c in df_orders.columns
])
non_null_df = non_null_counts.toPandas().T.reset_index()
non_null_df.columns = ['Column Name', 'Non-Null Count']
non_null_df

,Column Name,Non-Null Count
0,order_id,400500
1,customer_id,400500
2,order_date,400500
3,bs_year,400500
4,order_status,400500
5,payment_method,400500
6,shipping_city,400500
7,shipping_country,400500
8,discount_pct,400200


### Check which columns have null values

In [4]:
null_counts = df_orders.select([
    count(when(col(c).isNull(), c)).alias(c) for c in df_orders.columns
])
null_df = null_counts.toPandas().T.reset_index()
null_df.columns = ['Column Name', 'Null Count']
null_df

,Column Name,Null Count
0,order_id,0
1,customer_id,0
2,order_date,0
3,bs_year,0
4,order_status,0
5,payment_method,0
6,shipping_city,0
7,shipping_country,0
8,discount_pct,300


### Fix 1: Fill null discount_pct with 0.0
A missing discount means no discount was applied, so 0.0 is the correct default.

In [5]:
df_orders = df_orders.withColumn(
    "discount_pct",
    when(col("discount_pct").isNull(), lit(0.0))
    .otherwise(col("discount_pct"))
)

print("Null discount_pct remaining:", df_orders.filter(col("discount_pct").isNull()).count())

Null discount_pct remaining: 0


### Fix 2: Remove duplicate order_id rows
Each order_id should appear exactly once. We keep the first and drop the rest.

In [6]:
before = df_orders.count()
df_orders = df_orders.dropDuplicates(["order_id"])
after = df_orders.count()

print(f"Rows before dedup: {before}, after: {after}, removed: {before - after}")

Rows before dedup: 400500, after: 400000, removed: 500


### Fix 3: Convert order_date to a proper date type
The column is stored as a string. We cast it to DateType so date comparisons work correctly.

In [7]:
df_orders = df_orders.withColumn("order_date", to_date(col("order_date")))

# Check how many dates could not be parsed (would become null after cast)
print("Unparseable order_date values:", df_orders.filter(col("order_date").isNull()).count())

Unparseable order_date values: 0


### Check the schema to confirm data types

In [8]:
df_orders.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- bs_year: integer (nullable = true)
 |-- order_status: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- shipping_city: string (nullable = true)
 |-- shipping_country: string (nullable = true)
 |-- discount_pct: double (nullable = true)



### Final check: Verify null counts after all cleaning

In [9]:
final_nulls = df_orders.select([
    count(when(col(c).isNull(), c)).alias(c) for c in df_orders.columns
])
final_null_df = final_nulls.toPandas().T.reset_index()
final_null_df.columns = ['Column Name', 'Null Count']
final_null_df

,Column Name,Null Count
0,order_id,0
1,customer_id,0
2,order_date,0
3,bs_year,0
4,order_status,0
5,payment_method,0
6,shipping_city,0
7,shipping_country,0
8,discount_pct,0


### Export the cleaned orders table

In [10]:
df_orders.coalesce(1).write.csv("../cleaned_dataset/orders_cleaned.csv", header=True, mode="overwrite")
print("Orders table exported successfully to 'cleaned_dataset/orders_cleaned.csv'")

Orders table exported successfully to 'cleaned_dataset/orders_cleaned.csv'
